# 🤖 BÀI TẬP LỚN: HỌC MÁY VỚI DỮ LIỆU DẠNG BẢNG
**Môn học:** Học Máy (CO3001) — Học kỳ I, Năm học 2025–2026  
**Trường:** Đại học Bách Khoa, ĐHQG-HCM  
**Bộ môn:** Khoa học Máy tính  
**Giảng viên hướng dẫn:** TS. Lê Thành Sách  

---

| Họ và Tên | MSSV | Email | Github |
|:----------|:----:|:------|:--------|
| **Hoàng Xuân Bách** | 2352082 | bach.hoang2407khmt@hcmut.edu.vn | bachkhmt |
| **Nguyễn Việt Hùng** | 2352424 | hung.nguyensubin106@hcmut.edu.vn | hung.nguyensubin106@hcmut.edu.vn |
| **Đặng Mậu Anh Quân** | 2352983 | quan.dangmauanh@hcmut.edu.vn | dangmauanhquan@gmail.com |
| **Cao Lê Minh Khoa** | 2352550 | khoa.caoleminh@hcmut.edu.vn | khoalearningcode |
| **Phạm Hồ Minh Khoa** | 2352585 | khoa.phamhominh@hcmut.edu.vn | khoa.phamhominh@hcmut.edu.vn |

---

## 📌 Dataset: Adult Census Income
**Nguồn:** UCI Machine Learning Repository  
**Bài toán:** Binary Classification — Dự đoán thu nhập >50K hay ≤50K  
**Kích thước:** ~48,842 mẫu × 14 features  
**Đặc điểm:** Có missing values (ký hiệu `?`), có categorical features  

---
**📋 Mục lục:**
1. Cài đặt & Import thư viện
2. Tải và khám phá dữ liệu (EDA)
3. Tiền xử lý dữ liệu (Preprocessing)
4. Trích xuất & lựa chọn đặc trưng (Feature Engineering)
5. Huấn luyện mô hình (Model Training)
6. Đánh giá & So sánh kết quả (Evaluation)
7. *(Bonus)* Deep Learning Pipeline


---
# 1. 📦 Cài Đặt & Import Thư Viện

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from modules.preprocessing import load_data, preprocess_data
from modules.models import train_random_forest, evaluate_model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, time, joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, auc, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import category_encoders as ce

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 100})

# Tạo thư mục lưu kết quả
_NB_DIR  = os.path.dirname(os.path.abspath('__file__'))  # thư mục notebooks/
ROOT_DIR = os.path.abspath(os.path.join(_NB_DIR, '..'))  # project root

DATA_DIR      = os.path.join(ROOT_DIR, 'data')
RAW_DATA_DIR  = os.path.join(DATA_DIR, 'raw')        # data/raw/
PROC_DATA_DIR = os.path.join(DATA_DIR, 'processed')  # data/processed/
FEAT_DIR      = os.path.join(ROOT_DIR, 'features')
MOD_DIR       = os.path.join(ROOT_DIR, 'models')
REP_DIR       = os.path.join(ROOT_DIR, 'reports')
FIG_DIR       = os.path.join(REP_DIR,  'figures')

# Tạo thư mục nếu chưa có
for d in [DATA_DIR, RAW_DATA_DIR, PROC_DATA_DIR, FEAT_DIR, MOD_DIR, REP_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"📁 Project root : {ROOT_DIR}")
print(f"   data/raw/    : {RAW_DATA_DIR}")
print(f"   data/processed/: {PROC_DATA_DIR}")
print(f"   features/    : {FEAT_DIR}")
print(f"   models/      : {MOD_DIR}")
print(f"   reports/     : {REP_DIR}")

RANDOM_STATE = 42
print("✅ Import thư viện hoàn tất!")
print(f"   numpy   : {np.__version__}")
print(f"   pandas  : {pd.__version__}")
print(f"   sklearn : {__import__('sklearn').__version__}")

---
# 2. 📊 Tải Dữ Liệu & Khám Phá (EDA)

## 2.1 Tải Dataset từ UCI Repository

In [ ]:
# ── Download trực tiếp từ UCI — tự động cache local để tránh download lại ──
import urllib.request, os

TRAIN_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL   = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

# Tự động dùng thư mục hiện tại nếu '../data/' không tồn tại (Colab / chạy tại root)
DATA_DIR = '../data' if os.path.isdir('../data') else 'data'
os.makedirs(DATA_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(RAW_DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(RAW_DATA_DIR, 'adult_test.csv')

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# Chỉ download nếu chưa có file
if not os.path.exists(TRAIN_PATH):
    print('Đang tải dữ liệu từ UCI...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)
    print('✅ Tải xong!')
else:
    print(f'✅ Dùng file cache: {TRAIN_PATH}')

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS,
                        sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS,
                        sep=r',\s*', engine='python', na_values='?',
                        skiprows=1)

# Gộp train + test để EDA toàn bộ, sau đó split lại
df_full = pd.concat([df_train, df_test], ignore_index=True)

# Chuẩn hoá nhãn (adult.test có dấu '.' ở cuối)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()

print(f'✅ Dataset loaded! Train: {df_train.shape}, Test: {df_test.shape}')
print(f'   Tổng: {df_full.shape[0]:,} mẫu × {df_full.shape[1]} cột')
df_full.head()


## 2.2 Thông Tin Cơ Bản

In [ ]:
print("=" * 55)
print("THÔNG TIN CƠ BẢN")
print("=" * 55)
print(f"Số mẫu    : {df_full.shape[0]:,}")
print(f"Số features: {df_full.shape[1] - 1}")
print(f"Duplicates : {df_full.duplicated().sum():,}")
print()
df_full.info()

In [ ]:
print("THỐNG KÊ MÔ TẢ — Numeric Features")
df_full.describe().round(2)

In [ ]:
print("THỐNG KÊ MÔ TẢ — Categorical Features")
df_full.describe(include='object').T

## 2.3 Phân Tích Missing Values

In [ ]:
missing = df_full.isnull().sum()
missing_pct = 100 * missing / len(df_full)
missing_df = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct.round(2)})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

print("⚠️  Cột có missing values:")
print(missing_df)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing_df.index, missing_df['Percentage (%)'],
               color=['#e74c3c','#e67e22','#f1c40f'], edgecolor='white')
for bar, val in zip(bars, missing_df['Percentage (%)']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=11)
ax.set_xlabel('Missing (%)', fontsize=12)
ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Phân Tích Biến Mục Tiêu (Target)

In [ ]:
target_counts = df_full['income'].value_counts()
target_pct    = df_full['income'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['green', '#e74c3c']

# Bar chart
bars = axes[0].bar(target_counts.index, target_counts.values,
                   color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Phân phối Target (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số mẫu')
for bar, v in zip(bars, target_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+200, f'{v:,}',
                 ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 13})
axes[1].set_title('Phân phối Target (%)', fontsize=13, fontweight='bold')

plt.suptitle('Adult Census Income — Target Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

major_label = target_pct.idxmax()   # ten nhan
major_val = target_pct.max()        # % nhan

print(target_counts.to_string())
print()
print(f"→ Tập dữ liệu bị mất cân bằng (imbalanced): {major_label} chiếm ~{major_val:.2f}%")

## 2.5 Phân Tích Numeric Features

In [ ]:
numeric_cols = ['age','fnlwgt','education_num','capital_gain',
                'capital_loss','hours_per_week']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    # Histogram by class
    for cls, color in zip(['<=50K', '>50K'], ['#3498db','#e74c3c']):
        ax.hist(df_full[df_full['income']==cls][col].dropna(),
                bins=35, alpha=0.6, label=cls, color=color, edgecolor='none')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Count')

plt.suptitle('Phân phối Numeric Features theo Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'numeric_by_target.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots — phát hiện outliers
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    df_full.boxplot(column=col, by='income', ax=axes[idx],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7))
    axes[idx].set_title(col, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('')

plt.suptitle('Box Plots — Numeric Features by Income', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.6 Phân Tích Categorical Features

In [ ]:
cat_cols = ['workclass','education','marital_status','occupation',
            'relationship','race','sex','native_country']

# Top categories for each feature vs income
fig, axes = plt.subplots(4, 2, figsize=(18, 22))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    top_cats = df_full[col].value_counts().head(8).index
    subset   = df_full[df_full[col].isin(top_cats)]
    ct       = pd.crosstab(subset[col], subset['income'], normalize='index') * 100

    ct.plot(kind='barh', ax=axes[idx], color=['#3498db','#e74c3c'],
            edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('%')
    axes[idx].legend(title='Income', fontsize=9)

plt.suptitle('Categorical Features — % Income by Category', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'categorical_by_target.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Correlation Heatmap

In [ ]:
# Encode target tạm để tính correlation
df_corr = df_full[numeric_cols].copy()
df_corr['income_binary'] = (df_full['income'] == '>50K').astype(int)

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, mask=mask,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix (Numeric Features + Target)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("→ Nhận xét: education_num và age có tương quan dương với thu nhập >50K")

## 2.8 Tổng Kết EDA

In [ ]:
print("=" * 55)
print("TỔNG KẾT EDA")
print("=" * 55)
print(f"Tổng mẫu              : {len(df_full):,}")
print(f"Numeric features      : {len(numeric_cols)}")
print(f"Categorical features  : {len(cat_cols)}")
print(f"Cột có missing values : {len(missing_df)}")
print(f"  → workclass  : {missing_df.loc['workclass','Count']:,} ({missing_df.loc['workclass','Percentage (%)']:.2f}%)")
print(f"  → occupation : {missing_df.loc['occupation','Count']:,} ({missing_df.loc['occupation','Percentage (%)']:.2f}%)")
print(f"  → native_country: {missing_df.loc['native_country','Count']:,} ({missing_df.loc['native_country','Percentage (%)']:.2f}%)")
print(f"Duplicate rows        : {df_full.duplicated().sum():,}")   
print(f"Imbalanced target     : ≤50K = {target_pct['<=50K']:.1f}% | >50K = {target_pct['>50K']:.1f}%")
print()
print("📌 Vấn đề cần xử lý:")
print("  1. Missing values trong workclass, occupation, native_country")
print("  2. Imbalanced classes → dùng class_weight='balanced'")
print("  3. Outliers trong capital_gain, capital_loss, fnlwgt → RobustScaler")
print("  4. 8 categorical features → cần encoding")

---
# 3. 🔧 Tiền Xử Lý Dữ Liệu (Preprocessing)

## 3.1 Chuẩn Bị Data

In [ ]:
# Loại bỏ duplicate
df = df_full.drop_duplicates().reset_index(drop=True)
print(f"Sau khi bỏ duplicate: {df.shape}")

# Lưu processed data (đã gộp + chuẩn hoá nhãn + bỏ duplicate)
PROC_PATH = os.path.join(PROC_DATA_DIR, 'adult_full_clean.csv')
df.to_csv(PROC_PATH, index=False)
print(f"✅ Lưu processed data: {PROC_PATH}")

# Tách features và target
X = df.drop(columns=['income'])
y = (df['income'] == '>50K').astype(int)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class 0 (≤50K): {(y==0).sum():,}  |  Class 1 (>50K): {(y==1).sum():,}")

## 3.2 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Train class ratio: {y_train.mean():.3f} (>50K)")
print(f"Test  class ratio: {y_test.mean():.3f} (>50K)")

## 3.3 Định Nghĩa Preprocessing Pipeline (Có Thể Cấu Hình)

In [ ]:
# ── CÁC CỘT THEO LOẠI ──
numeric_features     = ['age','fnlwgt','education_num','capital_gain',
                        'capital_loss','hours_per_week']
categorical_features = ['workclass','education','marital_status','occupation',
                        'relationship','race','sex','native_country']

def build_preprocessor(impute_num='median', impute_cat='most_frequent',
                        encoding='onehot', scaling='standard'):
    """
    Xây dựng preprocessing pipeline có thể cấu hình.
    
    Parameters:
    -----------
    impute_num  : 'mean' | 'median' | 'knn'
    impute_cat  : 'most_frequent' | 'constant'
    encoding    : 'onehot' | 'label' | 'target'
    scaling     : 'standard' | 'minmax' | 'robust'
    """
    # ── Imputer ──
    if impute_num == 'knn':
        num_imputer = KNNImputer(n_neighbors=5)
    else:
        num_imputer = SimpleImputer(strategy=impute_num)
    cat_imputer = SimpleImputer(strategy=impute_cat,
                                fill_value='Unknown' if impute_cat=='constant' else None)
    
    # ── Scaler ──
    if scaling == 'standard':
        scaler = StandardScaler()
    elif scaling == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    else:
        scaler = RobustScaler()
    
    # ── Encoder ──
    if encoding == 'onehot':
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    elif encoding == 'label':
        # Label encoding via OrdinalEncoder
        from sklearn.preprocessing import OrdinalEncoder
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    else:
        # Target encoding — xử lý riêng bên ngoài Pipeline
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    
    numeric_pipeline = Pipeline([
        ('imputer', num_imputer),
        ('scaler',  scaler)
    ])
    categorical_pipeline = Pipeline([
        ('imputer', cat_imputer),
        ('encoder', encoder)
    ])
    
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline,     numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
    
    return preprocessor

print("✅ Hàm build_preprocessor() đã sẵn sàng!")
print("   Tham số có thể cấu hình:")
print("   - impute_num  : 'mean' | 'median' | 'knn'")
print("   - encoding    : 'onehot' | 'label'")
print("   - scaling     : 'standard' | 'minmax' | 'robust'")

## 3.4 So Sánh Các Cấu Hình Preprocessing

In [ ]:
from sklearn.linear_model import LogisticRegression

PREP_CONFIGS = {
    'Mean + OneHot + Standard' : dict(impute_num='mean',   encoding='onehot', scaling='standard'),
    'Median + OneHot + MinMax' : dict(impute_num='median', encoding='onehot', scaling='minmax'),
    'Median + Label + Robust'  : dict(impute_num='median', encoding='label',  scaling='robust'),
    'KNN + OneHot + Standard'  : dict(impute_num='knn',    encoding='onehot', scaling='standard'),
}

lr_probe = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, class_weight='balanced')

prep_results = {}
for name, cfg in PREP_CONFIGS.items():
    prep = build_preprocessor(**cfg)
    X_tr_p = prep.fit_transform(X_train, y_train)
    X_te_p = prep.transform(X_test)
    lr_probe.fit(X_tr_p, y_train)
    acc = lr_probe.score(X_te_p, y_test)
    prep_results[name] = {'accuracy': acc, 'n_features': X_tr_p.shape[1]}
    print(f"[{name}]  accuracy={acc:.4f}  features={X_tr_p.shape[1]}")

best_cfg_name = max(prep_results, key=lambda k: prep_results[k]['accuracy'])
print(f"\n✅ Cấu hình tốt nhất: {best_cfg_name}")

## 3.5. ✅ Áp Dụng Best Config & Lưu Features

In [ ]:
# ── Dùng cấu hình tốt nhất cho toàn bộ pipeline ──
BEST_CFG = PREP_CONFIGS[best_cfg_name]
preprocessor = build_preprocessor(**BEST_CFG)

X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"Shape sau preprocessing — Train: {X_train_proc.shape}, Test: {X_test_proc.shape}")

# Lưu split data gốc (trước preprocessing) vào data/processed/
X_train_raw_path = os.path.join(PROC_DATA_DIR, 'X_train_raw.csv')
X_test_raw_path  = os.path.join(PROC_DATA_DIR, 'X_test_raw.csv')
y_train_path     = os.path.join(PROC_DATA_DIR, 'y_train.csv')
y_test_path      = os.path.join(PROC_DATA_DIR, 'y_test.csv')
X_train.to_csv(X_train_raw_path, index=False)
X_test.to_csv(X_test_raw_path,   index=False)
y_train.to_csv(y_train_path,     index=False, header=['income'])
y_test.to_csv(y_test_path,       index=False, header=['income'])
print(f"✅ Lưu split raw: {PROC_DATA_DIR}/")

# Lưu preprocessed features
np.save(os.path.join(FEAT_DIR, 'X_train_preprocessed.npy'), X_train_proc)
np.save(os.path.join(FEAT_DIR, 'X_test_preprocessed.npy'),  X_test_proc)
np.save(os.path.join(FEAT_DIR,'y_train.npy'), y_train.values)
np.save(os.path.join(FEAT_DIR,'y_test.npy'),  y_test.values)
joblib.dump(preprocessor, os.path.join(MOD_DIR, 'preprocessor.pkl'))
print("✅ Đã lưu features và preprocessor!")

---
# 4. 🔬 Trích Xuất & Lựa Chọn Đặc Trưng

## 4.1 PCA — Giảm Số Chiều

In [ ]:
pca_results = {}
for vr in [0.90, 0.95, 0.99]:
    pca = PCA(n_components=vr, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_train_proc)
    pca_results[vr] = {
        'n_components': pca.n_components_,
        'explained_var': sum(pca.explained_variance_ratio_)
    }
    print(f"PCA {vr*100:.0f}%: {X_train_proc.shape[1]} → {pca.n_components_} components "
          f"(giữ {pca_results[vr]['explained_var']*100:.2f}% variance)")

# Vẽ scree plot
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_proc)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, len(cumvar)+1), cumvar, marker='o', markersize=3,
        color='royalblue', linewidth=2)
for thr, ls in [(90,'--'), (95,'-.')]:
    ax.axhline(thr, linestyle=ls, color='red', alpha=0.7, label=f'{thr}% threshold')
    idx = np.argmax(cumvar >= thr)
    ax.axvline(idx+1, linestyle=ls, color='orange', alpha=0.6)
ax.set_xlabel('Số components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance (%)', fontsize=12)
ax.set_title('PCA Scree Plot', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pca_scree_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dùng PCA 95% cho pipeline chính
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca_95.fit_transform(X_train_proc)
X_test_pca  = pca_95.transform(X_test_proc)

print(f"Sau PCA 95%: {X_train_proc.shape[1]} → {X_train_pca.shape[1]} features")

np.save(os.path.join(FEAT_DIR, 'X_train_pca.npy'), X_train_pca)
np.save(os.path.join(FEAT_DIR, 'X_test_pca.npy'),  X_test_pca)
joblib.dump(pca_95, os.path.join(MOD_DIR,'pca_95.pkl'))
print("✅ Lưu PCA features!")

## 4.2 Feature Importance — Random Forest

In [ ]:
rf_probe = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                  n_jobs=-1, class_weight='balanced')
rf_probe.fit(X_train_proc, y_train)

# Lấy tên features
try:
    ohe_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
    all_names = numeric_features + list(ohe_names)
except:
    all_names = [f'feature_{i}' for i in range(X_train_proc.shape[1])]

importances = rf_probe.feature_importances_
top_n       = min(20, len(all_names))
indices     = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(range(top_n), importances[indices], color='teal', alpha=0.8, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([all_names[i] for i in indices], fontsize=9)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n→ Top 5 features quan trọng nhất:")
for i in indices[:5]:
    print(f"   {all_names[i]}: {importances[i]:.4f}")

---
# 5. 🤖 Huấn Luyện Mô Hình (Model Training)

## 5.1 Định Nghĩa Các Mô Hình

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', probability=True,
        random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        use_label_encoder=False, eval_metric='logloss',
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum()
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        verbose=-1, class_weight='balanced'
    ),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight='balanced', max_depth=10
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=RANDOM_STATE
    ),
}
print(f'✅ Đã định nghĩa {len(MODELS)} mô hình:')
for name in MODELS:
    print(f'   • {name}')

## 5.2 Train & Evaluate Toàn Bộ Models

### 5.2.1 Raw Features

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name=''):
    """Train và trả về dict metrics."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:,1] if hasattr(model,'predict_proba') else None
    
    metrics = {
        'accuracy'  : accuracy_score(y_te, y_pred),
        'precision' : precision_score(y_te, y_pred, zero_division=0),
        'recall'    : recall_score(y_te, y_pred, zero_division=0),
        'f1'        : f1_score(y_te, y_pred, zero_division=0),
        'roc_auc'   : roc_auc_score(y_te, y_proba) if y_proba is not None else None,
        'train_time': round(train_time, 2)
    }
    return metrics

# ── Train trên raw preprocessed (không PCA) ──
print("Training trên preprocessed features (không PCA)...")
print("-" * 65)

results_raw = {}
trained_models = {}

for name, model in MODELS.items():
    m = evaluate_model(model, X_train_proc, y_train, X_test_proc, y_test, name)
    results_raw[name] = m
    trained_models[name] = model
    print(f"[{name:<22}] acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  "
          f"auc={m['roc_auc']:.4f}  time={m['train_time']}s")

results_raw_df = pd.DataFrame(results_raw).T
print("\n✅ Training hoàn tất!")

### 5.2.2 PCA Features

In [ ]:
# ── Train trên PCA features ──
print("Training trên PCA features (95% variance)...")
print("-" * 65)

results_pca = {}
for name, model in MODELS.items():
    import copy
    m_pca = copy.deepcopy(model)
    m = evaluate_model(m_pca, X_train_pca, y_train, X_test_pca, y_test, name)
    results_pca[name] = m
    print(f"[{name:<22}] acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  "
          f"auc={m['roc_auc']:.4f}  time={m['train_time']}s")

results_pca_df = pd.DataFrame(results_pca).T
print("\n✅ Training với PCA hoàn tất!")

## 5.3 So Sánh: Raw vs PCA Features

In [ ]:
compare_df = pd.DataFrame({
    'Accuracy (Raw)' : results_raw_df['accuracy'],
    'Accuracy (PCA)' : results_pca_df['accuracy'],
    'F1 (Raw)'       : results_raw_df['f1'],
    'F1 (PCA)'       : results_pca_df['f1'],
})
print("=== Raw vs PCA Features ===")
print(compare_df.round(4).sort_values('F1 (Raw)', ascending=False))

---
# 6. 📈 Đánh Giá & So Sánh Kết Quả

## 6.1 Bảng So Sánh Tổng Hợp

In [ ]:
results_df = results_raw_df.sort_values('f1', ascending=False)
print("=== KẾT QUẢ TRÊN TEST SET (Raw Features) ===")
print(results_df[['accuracy','precision','recall','f1','roc_auc','train_time']].round(4).to_string())

## 6.2 Biểu Đồ So Sánh Models

In [ ]:
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(22, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(results_df)))

for ax, metric in zip(axes, metrics_to_plot):
    sorted_df = results_df[metric].sort_values()
    bars = ax.barh(sorted_df.index, sorted_df.values,
                   color=colors, edgecolor='white', alpha=0.9)
    ax.set_xlabel(metric.upper(), fontsize=11)
    ax.set_title(metric.upper(), fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, sorted_df.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.3 Confusion Matrix — Top 3 Models

In [ ]:
top3_models = results_df.head(3).index.tolist()
fig, axes  = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, top3_models):
    model  = trained_models[name]
    y_pred = model.predict(X_test_proc)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(cm, display_labels=['≤50K', '>50K'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={results_df.loc[name,"accuracy"]:.4f}  F1={results_df.loc[name,"f1"]:.4f}',
                 fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrix — Top 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.4 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
palette = plt.cm.Set1(np.linspace(0, 0.85, len(results_df)))

for (name, _), color in zip(results_df.iterrows(), palette):
    model = trained_models[name]
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_proc)[:,1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, color=color,
                label=f'{name} (AUC={roc_auc_val:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1.5, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.5 Learning Curve — Best Model

In [ ]:
from sklearn.model_selection import learning_curve

best_name  = results_df.index[0]
best_model = trained_models[best_name]
print(f"Vẽ learning curve cho: {best_name}")

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_proc, y_train,
    cv=5, n_jobs=-1, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 8),
    random_state=RANDOM_STATE
)

tr_mean, tr_std = train_scores.mean(1), train_scores.std(1)
va_mean, va_std = val_scores.mean(1),   val_scores.std(1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, tr_mean, 'o-', color='royalblue', lw=2, label='Training F1')
ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.1, color='royalblue')
ax.plot(train_sizes, va_mean, 's-', color='coral',     lw=2, label='Validation F1')
ax.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.1, color='coral')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title(f'Learning Curve — {best_name}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'learning_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.6 Hyperparameter Tuning — Best Model

In [ ]:
# GridSearch cho Random Forest (hoặc model tốt nhất)
tune_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1,
                                    class_weight='balanced')
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 10, 20],
    'min_samples_split': [2, 5],
}

print("Đang chạy GridSearchCV... (có thể mất vài phút)")
grid_search = GridSearchCV(tune_model, param_grid, cv=3,
                           scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train_proc, y_train)

print(f"\nBest params : {grid_search.best_params_}")
print(f"Best CV F1  : {grid_search.best_score_:.4f}")

tuned_model  = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_proc)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Test F1      : {f1_score(y_test, y_pred_tuned):.4f}")

## 6.7 Lưu Kết Quả & Best Model

In [ ]:
# Lưu results
results_df.to_csv(os.path.join(REP_DIR, 'model_results.csv'))
print("✅ Lưu model_results.csv")

# Lưu best model
best_name  = results_df.index[0]
best_model = trained_models[best_name]
joblib.dump(best_model, os.path.join(MOD_DIR, f'best_model_{best_name.replace(" ","_")}.pkl'))
print(f"✅ Lưu best model: {best_name}")

print("\n" + "="*55)
print("TỔNG KẾT KẾT QUẢ")
print("="*55)
for name in results_df.index[:3]:
    r = results_df.loc[name]
    print(f"[{name:<22}] acc={r['accuracy']:.4f}  f1={r['f1']:.4f}  auc={r['roc_auc']:.4f}")
print(f"\n🏆 Best model: {best_name}")

---
# 7. 🧠 (Bonus) Deep Learning Pipeline

In [ ]:
TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
    print(f"✅ PyTorch: {torch.__version__}")
except ImportError:
    print("⚠️  PyTorch chưa cài. Bỏ qua phần Bonus.")

## 7.1 MLP Model (Nếu PyTorch Available)

In [ ]:
if TORCH_AVAILABLE:
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden=[256, 128, 64], dropout=0.3):
            super().__init__()
            layers = []
            prev = input_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                           nn.ReLU(), nn.Dropout(dropout)]
                prev = h
            layers.append(nn.Linear(prev, 2))
            self.net = nn.Sequential(*layers)
        def forward(self, x):
            return self.net(x)

    input_dim  = X_train_proc.shape[1]
    mlp_model  = MLP(input_dim)
    device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mlp_model  = mlp_model.to(device)
    print(f"MLP trên: {device}")
    print(mlp_model)
else:
    print("⏭️  Bỏ qua — PyTorch không khả dụng.")

In [ ]:
if TORCH_AVAILABLE:
    # Chuẩn bị data
    X_tr_t = torch.FloatTensor(X_train_proc).to(device)
    y_tr_t = torch.LongTensor(y_train.values).to(device)
    X_te_t = torch.FloatTensor(X_test_proc).to(device)

    dataset    = TensorDataset(X_tr_t, y_tr_t)
    loader     = DataLoader(dataset, batch_size=256, shuffle=True)
    criterion  = nn.CrossEntropyLoss(weight=torch.FloatTensor(
                     [1.0, (y_train==0).sum()/(y_train==1).sum()]).to(device))
    optimizer  = optim.Adam(mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    EPOCHS = 50
    history = {'loss': [], 'acc': []}

    for epoch in range(EPOCHS):
        mlp_model.train()
        total_loss, correct, total = 0, 0, 0
        for xb, yb in loader:
            optimizer.zero_grad()
            out  = mlp_model(xb)
            loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == yb).sum().item()
            total      += len(yb)
        scheduler.step()
        history['loss'].append(total_loss/len(loader))
        history['acc'].append(correct/total)
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss={history['loss'][-1]:.4f}  "
                  f"acc={history['acc'][-1]:.4f}")

    print("\n✅ MLP training hoàn tất!")

In [ ]:
if TORCH_AVAILABLE:
    # Đánh giá MLP
    mlp_model.eval()
    with torch.no_grad():
        out      = mlp_model(X_te_t)
        y_pred_mlp  = out.argmax(1).cpu().numpy()
        y_proba_mlp = torch.softmax(out, 1)[:,1].cpu().numpy()

    mlp_acc = accuracy_score(y_test, y_pred_mlp)
    mlp_f1  = f1_score(y_test, y_pred_mlp)
    mlp_auc = roc_auc_score(y_test, y_proba_mlp)
    print(f"MLP  →  Accuracy: {mlp_acc:.4f}  F1: {mlp_f1:.4f}  AUC: {mlp_auc:.4f}")

    # Training history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(history['loss'], color='royalblue', lw=2)
    ax1.set_title('MLP Training Loss', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
    ax2.plot(history['acc'], color='coral', lw=2)
    ax2.set_title('MLP Training Accuracy', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.grid(True, alpha=0.3)
    plt.suptitle('MLP Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'mlp_training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 7.2 So Sánh: Traditional ML vs Deep Learning

In [ ]:
if TORCH_AVAILABLE:
    # Gộp kết quả
    compare_final = results_df[['accuracy','f1','roc_auc']].copy()
    compare_final['type'] = 'Traditional ML'
    compare_final.loc['MLP'] = [mlp_acc, mlp_f1, mlp_auc, 'Deep Learning']
    compare_final[['accuracy','f1','roc_auc']] =         compare_final[['accuracy','f1','roc_auc']].astype(float)

    compare_final = compare_final.sort_values('f1', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 8))
    colors_bar = ['#e74c3c' if t == 'Deep Learning' else '#3498db'
                  for t in compare_final['type']]
    bars = ax.barh(compare_final.index, compare_final['f1'],
                   color=colors_bar, edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, compare_final['f1']):
        ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498db', label='Traditional ML'),
        Patch(facecolor='#e74c3c', label='Deep Learning (MLP)')
    ]
    ax.legend(handles=legend_elements, fontsize=11)
    ax.set_xlabel('F1 Score', fontsize=12)
    ax.set_title('Traditional ML vs Deep Learning — F1 Score', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'ml_vs_dl.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⏭️  Bỏ qua — PyTorch không khả dụng.")

---
# 8. 📋 Tổng Kết Toàn Bộ Notebook

In [ ]:
print("=" * 65)
print("TỔNG KẾT BÀI TẬP LỚN — HỌC MÁY VỚI TABULAR DATA")
print("=" * 65)
print()
print("📊 Dataset: Adult Census Income (UCI)")
print(f"   - {len(df):,} mẫu × {X.shape[1]} features")
print(f"   - Missing values: workclass, occupation, native_country")
print(f"   - Imbalanced: ≤50K={target_pct['<=50K']:.1f}% | >50K={target_pct['>50K']:.1f}%")
print()
print(f"🔧 Preprocessing (best config): {best_cfg_name}")
print(f"   Features sau xử lý: {X_train_proc.shape[1]}")
print(f"   Features sau PCA 95%: {X_train_pca.shape[1]}")
print()
print("🤖 Kết quả Top 3 Models:")
for name in results_df.index[:3]:
    r = results_df.loc[name]
    print(f"   [{name:<22}] acc={r['accuracy']:.4f}  f1={r['f1']:.4f}  auc={r['roc_auc']:.4f}")
print()
print("💾 Files đã lưu:")
print(f"   {RAW_DATA_DIR}/adult_train.csv  |  adult_test.csv")
print(f"   {PROC_DATA_DIR}/adult_full_clean.csv")
print(f"   {PROC_DATA_DIR}/X_train_raw.csv  |  X_test_raw.csv")
print(f"   {PROC_DATA_DIR}/y_train.csv  |  y_test.csv")
print(f"   {FEAT_DIR}/X_train_preprocessed.npy  |  X_test_preprocessed.npy")
print(f"   {FEAT_DIR}/X_train_pca.npy  |  X_test_pca.npy")
print(f"   {FEAT_DIR}/y_train.npy  |  y_test.npy")
print(f"   {MOD_DIR}/preprocessor.pkl  |  pca_95.pkl")
print(f"   {MOD_DIR}/best_model_{best_name.replace(' ','_')}.pkl")
print(f"   {REP_DIR}/model_results.csv")
print(f"   {FIG_DIR}/*.png")
print()
print("✅ Notebook chạy thành công!")